# Navier–Stokes baseline vs physics runs

Run the three experiments (baseline, physics/fixed, physics/adaptive), capture metrics, and save comparison plots. The defaults are set for a quick demo; bump `FAST_DEMO` to `False` for longer training once you have GPU time and the dataset in place.

In [ ]:
# If running in Colab for the first time, uncomment the next line to install the package in editable mode.
# %pip install -q -e .

In [ ]:
import os
import json
import pathlib
import random
from copy import deepcopy
from datetime import datetime
from typing import Dict, Any

import torch
import pandas as pd
import matplotlib.pyplot as plt

from config.navier_stokes_config import Default
from neuralop import H1Loss, LpLoss, get_model
from neuralop.losses.equation_losses import NavierStokesEqnLoss
from neuralop.data.datasets.navier_stokes import load_navier_stokes_pt
from neuralop.data.transforms.data_processors import MGPatchingDataProcessor
from neuralop.training import AdamW, PhysicsWeightScheduler
from neuralop.training.trainer import Trainer

PROJECT_ROOT = pathlib.Path(".").resolve()
# Update here if your dataset lives elsewhere (e.g., on Google Drive)
DATA_ROOT = pathlib.Path("~/data/navier_stokes").expanduser()
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "navier_stokes_colab"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Quick demo vs longer run toggle
FAST_DEMO = True  # set to False for a longer training run
DEMO_EPOCHS = 3
FULL_EPOCHS = 50  # raise to 600 for full training if you have time/GPU
N_EPOCHS = DEMO_EPOCHS if FAST_DEMO else FULL_EPOCHS
N_TRAIN = 256 if FAST_DEMO else 2000
BATCH_SIZE = 4 if FAST_DEMO else 8
TEST_BATCH_SIZE = 4 if FAST_DEMO else 8
TEST_RESOLUTIONS = [128]
EVAL_INTERVAL = 1

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Output dir:   {OUTPUT_ROOT}")
print(f"Device:       {device}")
print(f"Running {N_EPOCHS} epochs, n_train={N_TRAIN}, batch_size={BATCH_SIZE}")

In [ ]:
def _to_float(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().item()
    if isinstance(x, (int, float)):
        return float(x)
    return x


class LoggingTrainer(Trainer):
    """Thin wrapper to keep per-epoch train/eval metrics in memory."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_history = []
        self.eval_history = []

    def train_one_epoch(self, epoch, train_loader, training_loss):
        train_err, avg_loss, avg_lasso_loss, epoch_train_time = super().train_one_epoch(
            epoch, train_loader, training_loss
        )
        record = {
            "epoch": epoch,
            "train_err": _to_float(train_err),
            "avg_loss": _to_float(avg_loss),
            "avg_lasso_loss": _to_float(avg_lasso_loss) if avg_lasso_loss is not None else None,
            "time": _to_float(epoch_train_time),
        }
        for k, v in self.latest_train_metrics.items():
            if isinstance(v, dict):
                record[k] = {kk: _to_float(vv) for kk, vv in v.items()}
            else:
                record[k] = _to_float(v)
        self.train_history.append(record)
        return train_err, avg_loss, avg_lasso_loss, epoch_train_time

    def evaluate_all(self, epoch, eval_losses, test_loaders, eval_modes, max_autoregressive_steps=None):
        metrics = super().evaluate_all(epoch, eval_losses, test_loaders, eval_modes, max_autoregressive_steps)
        if epoch is not None:
            record = {"epoch": epoch}
            for k, v in metrics.items():
                record[k] = _to_float(v)
            self.eval_history.append(record)
        return metrics


def build_scheduler(cfg, optimizer):
    if cfg.opt.scheduler == "ReduceLROnPlateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, factor=cfg.opt.gamma, patience=cfg.opt.scheduler_patience, mode="min"
        )
    if cfg.opt.scheduler == "CosineAnnealingLR":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.opt.scheduler_T_max)
    if cfg.opt.scheduler == "StepLR":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg.opt.step_size, gamma=cfg.opt.gamma)
    raise ValueError(f"Unknown scheduler {cfg.opt.scheduler}")


def run_experiment(run_name: str, physics_cfg: Dict[str, Any]):
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT {DATA_ROOT} not found. Mount/copy your Navier–Stokes dataset there or update DATA_ROOT.")

    cfg = Default()
    cfg.wandb.log = False
    cfg.verbose = True
    cfg.distributed.use_distributed = False
    cfg.patching.levels = 0

    cfg.data.folder = str(DATA_ROOT)
    cfg.data.n_train = N_TRAIN
    cfg.data.batch_size = BATCH_SIZE
    cfg.data.train_resolution = TEST_RESOLUTIONS[0]
    cfg.data.test_resolutions = TEST_RESOLUTIONS
    cfg.data.test_batch_sizes = [TEST_BATCH_SIZE for _ in TEST_RESOLUTIONS]
    cfg.data.n_tests = [min(128, cfg.data.n_tests[0])]  # keep eval light in demo

    cfg.opt.n_epochs = N_EPOCHS
    cfg.opt.eval_interval = EVAL_INTERVAL
    cfg.opt.mixed_precision = False

    cfg.physics_loss.enabled = physics_cfg.get("enabled", False)
    cfg.physics_loss.weight_schedule = physics_cfg.get("weight_schedule", "none")
    cfg.physics_loss.initial_weight = physics_cfg.get("initial_weight", 0.0)
    cfg.physics_loss.max_weight = physics_cfg.get("max_weight", 1.0)
    cfg.physics_loss.warmup_epochs = min(cfg.opt.n_epochs, physics_cfg.get("warmup_epochs", cfg.physics_loss.warmup_epochs))
    cfg.physics_loss.denormalize = physics_cfg.get("denormalize", cfg.physics_loss.denormalize)
    cfg.physics_loss.derivative_mode = physics_cfg.get("derivative_mode", cfg.physics_loss.derivative_mode)

    train_loader, test_loaders, data_processor = load_navier_stokes_pt(
        data_root=pathlib.Path(cfg.data.folder),
        train_resolution=cfg.data.train_resolution,
        n_train=cfg.data.n_train,
        batch_size=cfg.data.batch_size,
        test_resolutions=cfg.data.test_resolutions,
        n_tests=cfg.data.n_tests,
        test_batch_sizes=cfg.data.test_batch_sizes,
        encode_input=cfg.data.encode_input,
        encode_output=cfg.data.encode_output,
        num_workers=2,
    )

    model = get_model(cfg).to(device)
    if cfg.patching.levels > 0:
        data_processor = MGPatchingDataProcessor(
            model=model,
            in_normalizer=data_processor.in_normalizer,
            out_normalizer=data_processor.out_normalizer,
            padding_fraction=cfg.patching.padding,
            stitching=cfg.patching.stitching,
            levels=cfg.patching.levels,
            use_distributed=False,
        )
    data_processor = data_processor.to(device)

    optimizer = AdamW(model.parameters(), lr=cfg.opt.learning_rate, weight_decay=cfg.opt.weight_decay)
    scheduler = build_scheduler(cfg, optimizer)

    l2loss = LpLoss(d=2, p=2)
    h1loss = H1Loss(d=2)
    train_loss = h1loss if cfg.opt.training_loss == "h1" else l2loss
    eval_losses = {"h1": h1loss, "l2": l2loss}

    phy_loss_fn = None
    phy_scheduler = None
    physics_weight = cfg.physics_loss.initial_weight
    phy_normalizer_train = None
    phy_normalizer_eval = None
    if cfg.physics_loss.enabled:
        phy_loss_fn = NavierStokesEqnLoss(
            viscosity=cfg.physics_loss.viscosity,
            dx=cfg.physics_loss.dx,
            dy=cfg.physics_loss.dy,
            dt=cfg.physics_loss.dt,
            advection_weight=cfg.physics_loss.advection_weight,
            diffusion_weight=cfg.physics_loss.diffusion_weight,
            forcing_weight=cfg.physics_loss.forcing_weight,
            use_vorticity_form=cfg.physics_loss.use_vorticity_form,
            derivative_mode=cfg.physics_loss.derivative_mode,
            denormalize=cfg.physics_loss.denormalize,
        )
        if cfg.physics_loss.weight_schedule in ["linear_warmup", "plateau"]:
            phy_scheduler = PhysicsWeightScheduler(
                initial_weight=cfg.physics_loss.initial_weight,
                max_weight=cfg.physics_loss.max_weight,
                warmup_epochs=cfg.physics_loss.warmup_epochs,
                mode=cfg.physics_loss.weight_schedule,
            )
        if cfg.physics_loss.denormalize and not isinstance(data_processor, MGPatchingDataProcessor):
            phy_normalizer_train = getattr(data_processor, "out_normalizer", None)
        phy_normalizer_eval = None if not isinstance(data_processor, MGPatchingDataProcessor) else phy_normalizer_train

    trainer = LoggingTrainer(
        model=model,
        n_epochs=cfg.opt.n_epochs,
        data_processor=data_processor,
        device=device,
        mixed_precision=cfg.opt.mixed_precision,
        eval_interval=cfg.opt.eval_interval,
        log_output=False,
        use_distributed=False,
        verbose=True,
        wandb_log=False,
    )

    final_metrics = trainer.train(
        train_loader,
        test_loaders,
        optimizer,
        scheduler,
        regularizer=False,
        training_loss=train_loss,
        eval_losses=eval_losses,
        physics_loss_fn=phy_loss_fn,
        physics_weight=physics_weight,
        physics_weight_scheduler=phy_scheduler,
        physics_normalizer_train=phy_normalizer_train,
        physics_normalizer_eval=phy_normalizer_eval,
        physics_return_components=True,
        eval_physics_loss=bool(phy_loss_fn),
    )

    result = {
        "name": run_name,
        "config": cfg.to_dict() if hasattr(cfg, "to_dict") else cfg.__dict__,
        "train_history": trainer.train_history,
        "eval_history": trainer.eval_history,
        "final_metrics": {k: _to_float(v) for k, v in final_metrics.items()},
    }
    torch.cuda.empty_cache()
    return result


In [ ]:
experiments = [
    {
        "name": "baseline",
        "physics": {
            "enabled": False,
            "weight_schedule": "none",
            "initial_weight": 0.0,
            "max_weight": 0.0,
        },
    },
    {
        "name": "phy_fixed",
        "physics": {
            "enabled": True,
            "weight_schedule": "none",
            "initial_weight": 1.0,
            "max_weight": 1.0,
        },
    },
    {
        "name": "phy_adaptive",
        "physics": {
            "enabled": True,
            "weight_schedule": "linear_warmup",
            "initial_weight": 0.1,
            "max_weight": 1.0,
            "warmup_epochs": max(1, N_EPOCHS // 2),
        },
    },
]

all_results = []
for exp in experiments:
    print(f"\n=== Running {exp['name']} ===")
    res = run_experiment(exp["name"], exp["physics"])
    all_results.append(res)
    print(f"Finished {exp['name']}, last metrics: {res['final_metrics']}")

In [ ]:
def save_results(results):
    for r in results:
        fname = OUTPUT_ROOT / f"{r['name']}_metrics.json"
        with open(fname, "w") as f:
            json.dump(r, f, indent=2)
        print(f"Saved {fname}")


save_results(all_results)
print(f"Artifacts saved under {OUTPUT_ROOT}")

In [ ]:
plt.style.use("seaborn-v0_8")


def plot_train_metric(results, key, ylabel, fname):
    plt.figure(figsize=(7, 4))
    plotted = False
    for r in results:
        df = pd.DataFrame(r["train_history"])
        if key in df:
            plt.plot(df["epoch"], df[key], label=r["name"])
            plotted = True
    if not plotted:
        plt.close()
        return
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.legend()
    out_path = OUTPUT_ROOT / fname
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved {out_path}")


def plot_eval_metric(results, key, ylabel, fname):
    plt.figure(figsize=(7, 4))
    plotted = False
    for r in results:
        df = pd.DataFrame(r["eval_history"])
        if key in df:
            plt.plot(df["epoch"], df[key], label=r["name"])
            plotted = True
    if not plotted:
        plt.close()
        return
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.legend()
    out_path = OUTPUT_ROOT / fname
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved {out_path}")


def plot_last_eval_bar(results, keys, fname):
    rows = []
    for r in results:
        if not r["eval_history"]:
            continue
        last = r["eval_history"][-1]
        rows.append({"run": r["name"], **{k: last.get(k, None) for k in keys}})
    if not rows:
        return
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 4))
    idx = range(len(df))
    bar_width = 0.8 / max(1, len(keys))
    for i, k in enumerate(keys):
        ax.bar([j + i * bar_width for j in idx], df[k], width=bar_width, label=k)
    ax.set_xticks([j + bar_width * (len(keys) - 1) / 2 for j in idx])
    ax.set_xticklabels(df["run"].tolist())
    ax.set_ylabel("metric value")
    ax.legend()
    out_path = OUTPUT_ROOT / fname
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved {out_path}")


# Training curves
plot_train_metric(all_results, "avg_data_loss", "avg_data_loss (train)", "train_data_loss.png")
plot_train_metric(all_results, "avg_physics_loss", "avg_physics_loss (train)", "train_physics_loss.png")
plot_train_metric(all_results, "physics_weight", "physics weight", "train_physics_weight.png")

# Eval curves (per resolution/metric key)
plot_eval_metric(all_results, "128_l2", "128_l2 (eval)", "eval_128_l2.png")
plot_eval_metric(all_results, "128_h1", "128_h1 (eval)", "eval_128_h1.png")
plot_eval_metric(all_results, "128_physics", "128_physics loss", "eval_128_physics.png")
plot_eval_metric(all_results, "128_physics_loss_residual", "128 residual loss", "eval_128_residual.png")

# Bar chart of last eval snapshot
plot_last_eval_bar(all_results, ["128_l2", "128_h1", "128_physics", "128_physics_loss_residual"], "eval_last_snapshot.png")

In [ ]:
print("Done. Files in output dir:")
for p in sorted(OUTPUT_ROOT.glob("*")):
    print("-", p)